# Chapter 5 — Optimizer Zoo
**Math for ML Engineers** · companion notebook

This notebook mirrors §5.3–5.5 (SGD, Momentum, Adam) on an ill-conditioned quadratic where the curvature ratio is 10×. The same landscape drives every Transformer training run; the pathology is visible in two dimensions.

**Equations implemented:** gradient descent update (5.1), momentum update (5.17)–(5.18), Adam bias-corrected update (5.36)–(5.37).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec


## The loss landscape

We use $f(\theta) = \frac{1}{2}(10\,\theta_1^2 + \theta_2^2)$, an axis-aligned quadratic with condition number $\kappa = 10$.  The gradient is $\nabla f = (10\theta_1,\, \theta_2)$.

A Transformer loss surface locally looks like this near a saddle: high curvature in some parameter directions (attention projection weights) and low curvature in others (embedding rows for rare tokens).

In [ ]:
def f(t):      return 0.5 * (10*t[0]**2 + t[1]**2)
def grad(t):   return np.array([10*t[0], t[1]])

theta0 = np.array([1.0, 1.0])
N = 300


## Optimiser implementations

Each optimiser is a pure-NumPy loop so the update equations are transparent.

In [ ]:
def run_sgd(lr=0.08, n=N):
    t, hist = theta0.copy(), [theta0.copy()]
    for _ in range(n):
        t = t - lr * grad(t)
        hist.append(t.copy())
    return np.array(hist)

def run_momentum(lr=0.08, beta=0.9, n=N):
    t, v, hist = theta0.copy(), np.zeros(2), [theta0.copy()]
    for _ in range(n):
        v = beta*v + (1-beta)*grad(t)   # EMA form (Adam-style)
        t = t - lr*v
        hist.append(t.copy())
    return np.array(hist)

def run_adam(lr=0.3, b1=0.9, b2=0.999, eps=1e-8, n=N):
    t, m, v, hist = theta0.copy(), np.zeros(2), np.zeros(2), [theta0.copy()]
    for step in range(1, n+1):
        g = grad(t)
        m = b1*m + (1-b1)*g
        v = b2*v + (1-b2)*g**2
        mh = m / (1 - b1**step)   # bias correction
        vh = v / (1 - b2**step)
        t = t - lr * mh / (np.sqrt(vh) + eps)
        hist.append(t.copy())
    return np.array(hist)

hist_sgd  = run_sgd()
hist_mom  = run_momentum()
hist_adam = run_adam()


## Visualisation

Left: trajectories on the loss contour.  Right: loss vs. step.

SGD oscillates across the narrow valley; momentum overshoots less; Adam adapts per-coordinate and converges fastest on this ill-conditioned surface — exactly the argument for using $\beta_2 = 0.95$ (not 0.999) for Transformers (§5.5).

In [ ]:
th1 = np.linspace(-1.1, 1.1, 300)
th2 = np.linspace(-1.1, 1.1, 300)
T1, T2 = np.meshgrid(th1, th2)
Z = 0.5*(10*T1**2 + T2**2)

fig = plt.figure(figsize=(13, 5))
gs = GridSpec(1, 2, figure=fig, wspace=0.35)

ax1 = fig.add_subplot(gs[0])
ax1.contour(T1, T2, Z, levels=np.logspace(-2, 1, 20), colors='gray', alpha=0.4)
for hist, lbl, col in [(hist_sgd,'SGD','C0'), (hist_mom,'Momentum','C1'), (hist_adam,'Adam','C2')]:
    ax1.plot(hist[:,0], hist[:,1], '-', color=col, lw=1.2, label=lbl)
    ax1.scatter(hist[0,0], hist[0,1], s=40, color=col)
ax1.set_xlabel(r'$\theta_1$'); ax1.set_ylabel(r'$\theta_2$')
ax1.set_title('Trajectories (κ = 10)')
ax1.legend()

ax2 = fig.add_subplot(gs[1])
for hist, lbl, col in [(hist_sgd,'SGD','C0'), (hist_mom,'Momentum','C1'), (hist_adam,'Adam','C2')]:
    losses = [f(t) for t in hist]
    ax2.semilogy(losses, color=col, lw=1.5, label=lbl)
ax2.set_xlabel('Step'); ax2.set_ylabel('f(θ) — log scale')
ax2.set_title('Loss convergence')
ax2.legend()
plt.suptitle('Ch.5 Optimiser Zoo — ill-conditioned quadratic (κ=10)', y=1.01)
plt.show()


## Adam β₂ sensitivity — the Transformer default

Standard Adam uses $\beta_2 = 0.999$.  For Transformers, $\beta_2 = 0.95$ is the production default (Llama-3, GPT-4 training recipes) because the second-moment window is shorter, adapting faster to heterogeneous gradient statistics across attention vs. embedding layers.

The plot below shows Adam with both values on the same quadratic.

In [ ]:
hist_999 = run_adam(b2=0.999)
hist_095 = run_adam(b2=0.95)

plt.figure(figsize=(6,4))
plt.semilogy([f(t) for t in hist_999], 'C3--', lw=1.5, label='β₂=0.999 (library default)')
plt.semilogy([f(t) for t in hist_095], 'C2-',  lw=1.5, label='β₂=0.95  (Transformer default)')
plt.xlabel('Step'); plt.ylabel('f(θ)')
plt.title('Adam β₂ sensitivity')
plt.legend(); plt.tight_layout(); plt.show()
print('Steps to reach f < 1e-3:')
for name, h in [('β₂=0.999', hist_999), ('β₂=0.95 ', hist_095)]:
    idx = next((i for i,t in enumerate(h) if f(t)<1e-3), None)
    print(f'  {name}: {idx} steps')


## Exercise

1. Re-run with `lr=0.01` for SGD. How many steps to reach $f < 10^{-3}$?  What does this illustrate about the learning-rate–condition-number interaction (§5.2, Theorem 5.4)?
2. Try `b1=0.0` in Adam (no first moment). Convergence changes dramatically. Why? Which term in the Adam update is responsible for the fast early progress?